# 01. Dataset Quality & Distribution Audit
**Project**: Recut Research Lab 01  
**Dataset**: CyberAgent AdTEC `ad-consistency` (NAACL 2025)  
**License**: CC BY-NC-SA 4.0  

### Research Objective
Audit data integrity, label distributions, text lengths, and cross-split overlap across the official `train`, `valid`, and `test` splits of the CyberAgent AdTEC ad-consistency benchmark.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from recut_lab.audit import format_audit_markdown, run_dataset_audit
from recut_lab.data import create_synthetic_dataset, load_adtec_dataset

data_dir = Path('../data/raw/adtec_consistency')
if (data_dir / 'train.jsonl').exists():
    print(f'[+] Loading real dataset from {data_dir.resolve()}')
    dataset = load_adtec_dataset(data_dir)
else:
    print('[!] Real dataset not found in data/raw/adtec_consistency. Using synthetic dataset for demonstration.')
    dataset = create_synthetic_dataset(num_samples_per_split=50)

print(f'Split sizes: {dataset.split_sizes()}')

In [ ]:
# Execute comprehensive audit
summary = run_dataset_audit(dataset)
display(Markdown(format_audit_markdown(summary)))

In [ ]:
# Visualize label distribution across splits
fig, ax = plt.subplots(figsize=(8, 4))
split_names = list(summary.splits.keys())
consistent_counts = [summary.splits[s].label_distribution.get('consistent', 0) for s in split_names]
inconsistent_counts = [summary.splits[s].label_distribution.get('inconsistent', 0) for s in split_names]

x = range(len(split_names))
width = 0.35
ax.bar([i - width/2 for i in x], consistent_counts, width, label='Consistent', color='#2b5c8f')
ax.bar([i + width/2 for i in x], inconsistent_counts, width, label='Inconsistent', color='#d95f02')
ax.set_ylabel('Count')
ax.set_title('Label Distribution Across Splits')
ax.set_xticks(list(x))
ax.set_xticklabels(split_names)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualize text length distributions (character counts)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for split_name, df in [('train', dataset.train), ('valid', dataset.valid)]:
    axes[0].hist(df['title'].str.len(), bins=30, alpha=0.6, label=split_name)
    axes[1].hist(df['lp_text'].str.len(), bins=30, alpha=0.6, label=split_name)

axes[0].set_title('Title Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].legend()

axes[1].set_title('Landing Page Character Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].legend()

plt.tight_layout()
plt.show()

### Audit Findings & Recommendations
1. **Label Balance**: Assess whether class imbalance requires balanced loss weighting or macro-averaged metrics.
2. **Text Lengths**: Character lengths for landing pages show wide dispersion compared to concise ad headlines.
3. **Data Leakage Risk**: Check cross-split overlap findings above to confirm whether shared landing pages exist across splits.